# OWL &rarr; Wikibase Synchronization Pipeline

A FAIR-oriented, restartable, idempotent pipeline that imports and
synchronizes an OWL ontology (`data/slr_reviewed.owl`) into a Wikibase
instance.

## What this notebook does

1. Parses the OWL/RDF graph directly with `rdflib` (the graph is the
   **source of truth** for the whole run -- no intermediate CSV).
2. Classifies every resource (classes, named individuals, object/annotation
   properties, blank nodes) and extracts labels, descriptions, aliases, and
   outgoing statements per resource.
3. Validates the user-maintained `PROPERTY_MAP` (OWL predicate &rarr;
   Wikibase PID/datatype) and the ontology's data quality *before* touching
   Wikibase.
4. Builds a single **synchronization plan** (two passes: resolve/create
   items, then add statements) that is used identically by dry-run and live
   execution.
5. In dry-run mode (the default), displays and exports the plan without
   writing anything. In live mode, executes exactly that plan.
6. Runs post-synchronization validation and writes machine-readable reports.

## Supported entity types

OWL classes and named individuals become Wikibase items. OWL object,
datatype, and annotation *property declarations* are not synchronized as
items -- they are the source of the `PROPERTY_MAP` configuration surface
instead (Section 4).

## Idempotency strategy

Every class/individual is identified by a **canonical ontology identifier**
(see `src/identifiers.py`), never by its label. Before creating an item, the
pipeline checks (1) the local JSON cache, then (2) Wikibase itself via the
configured identity property (`ontology_iri` by default). Statements are
only added if an equivalent value is not already present on the target item.
Re-running the notebook against an unchanged OWL file produces zero new
writes.

## Cache strategy

`cache/entity_lookup.json` maps canonical ontology ids to Wikibase QIDs and
is updated atomically immediately after every successful item creation.
`cache/synchronization_state.json` checkpoints run progress (phase reached,
processed entities, failed actions) so an interrupted run can be diagnosed
and safely resumed -- see Section 10 and the README's "Interruption
Recovery" section.

## Dry-run behavior

`DRY_RUN = True` by default (Section 3). Dry-run mode inspects the ontology
and, where credentials/endpoints allow, the *current* Wikibase state, then
computes and exports the exact plan live mode would execute -- but performs
zero writes and never mutates the production cache with invented QIDs.

## Synchronization policy

Defaults are conservative: labels/descriptions/aliases/statements are only
**added or updated**, never deleted, unless the corresponding
`REMOVE_OBSOLETE_*` / `DELETE_MISSING_ITEMS` flags are explicitly enabled
(they default to `False`). See Section 3 and the README's "Update and
Deletion Policy".

## Limitations and assumptions

- Qualifiers and references on statements are not yet synchronized (the
  data model has room for them -- see `src/models.py` -- but the planner
  does not populate them in this version).
- `UPDATE_STATEMENT` (in-place edits to an existing claim's value) is not
  implemented; the pipeline only adds missing statements. Changing a value
  in the OWL file adds a new statement rather than replacing the old one
  unless `REMOVE_OBSOLETE_STATEMENTS` support is extended.
- Remote duplicate detection requires a SPARQL endpoint and a PID for the
  identity property; without both, the pipeline still runs safely but can
  only detect duplicates that are already in the local cache.
- Wikibase datatypes must be configured explicitly in `PROPERTY_MAP`; the
  pipeline never guesses or invents a PID.

## Section 2: Imports and Environment Setup

Import third-party and standard-library packages, add the project's `src/`
package to the Python path, and load secrets from `.env`. Credentials are
**never** hardcoded in this notebook -- see `.env.example` for the variables
`WikibaseCredentials` expects.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import importlib
REQUIRED_PACKAGES = ["rdflib", "pandas", "dotenv", "wikibaseintegrator", "requests", "tenacity", "tqdm"]
missing = []
for package_name in REQUIRED_PACKAGES:
    try:
        importlib.import_module(package_name)
    except ImportError:
        missing.append(package_name)
if missing:
    raise ImportError(
        f"Missing required packages: {missing}. Install with: pip install -r requirements.txt"
    )
print("All required packages are importable:", REQUIRED_PACKAGES)

All required packages are importable: ['rdflib', 'pandas', 'dotenv', 'wikibaseintegrator', 'requests', 'tenacity', 'tqdm']


In [2]:
import json
import uuid
from datetime import datetime, timezone

import pandas as pd

from src.cache_manager import EntityLookupCache, SynchronizationState
from src.config import PROPERTY_MAP, PipelineConfig, WikibaseCredentials, build_property_mappings, load_wikibase_credentials
from src.datatype_conversion import convert_owl_value_to_wikibase
from src.hashing import compute_file_hash
from src.logging_setup import configure_logging
from src.models import ActionType, Severity
from src.ontology_parser import (
    ParserConfig,
    build_property_label_index,
    collect_syncable_entities,
    load_ontology_graph,
)
from src.reporting import (
    compute_ontology_statistics,
    save_dry_run_plan,
    save_ontology_statistics,
    save_synchronization_report,
    save_unresolved_resources,
    save_validation_report,
)
from src.sync_planner import build_synchronization_plan
from src.synchronizer import execute_synchronization
from src.validators import (
    collect_predicate_usage,
    has_blocking_findings,
    summarize_findings,
    validate_ontology_quality,
    validate_post_synchronization,
    validate_property_map,
)
from src.wikibase_client import WikibaseClient

RUN_ID = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')}-{uuid.uuid4().hex[:8]}"
RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat()
print(f"run_id = {RUN_ID}")
print(f"run_timestamp = {RUN_TIMESTAMP}")

run_id = 20260713T192017-8fa7c2dc
run_timestamp = 2026-07-13T19:20:17.697110+00:00


## Section 3: Central Configuration

Every setting that affects run *behavior* lives here, in one place, as a
single `PipelineConfig` instance (see `src/config.py`). Nothing later in the
notebook reads an environment variable or a magic constant directly -- it
all flows from `CONFIG`.

**`DRY_RUN = True` is the default and the safe starting point.** Only flip it
to `False` after reviewing the dry-run reports in Section 13.

In [3]:
CONFIG = PipelineConfig(
    owl_file_path=PROJECT_ROOT / "data" / "slr_reviewed.owl",
    cache_dir=PROJECT_ROOT / "cache",
    report_dir=PROJECT_ROOT / "reports",
    log_dir=PROJECT_ROOT / "logs",

    dry_run=True,  # <-- safety default. Flip only after inspecting the dry-run reports.

    sleep_time_seconds=0.5,
    max_retries=5,
    backoff_multiplier=2.0,
    request_timeout_seconds=30,
    batch_size=50,

    default_language="en",
    supported_languages=("en", "de"),

    update_labels=True,
    update_descriptions=True,
    update_aliases=True,
    update_statements=True,

    remove_obsolete_aliases=False,
    remove_obsolete_statements=False,
    delete_missing_items=False,
    metadata_sync_mode="merge",  # "merge" | "replace_managed_values" | "report_only"

    stop_on_unresolved_property=False,
    stop_on_validation_error=True,

    alias_local_names=("aliase", "alias", "aliases"),
    identity_property_key="ontology_iri",
)

CONFIG.ensure_directories()
LOGGER = configure_logging(CONFIG.log_dir, RUN_ID)
LOGGER.info("Pipeline configuration loaded. dry_run=%s", CONFIG.dry_run)

pd.DataFrame([{"setting": k, "value": str(v)} for k, v in CONFIG.sync_relevant_dict().items()])

2026-07-13T21:20:17 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync | Pipeline configuration loaded. dry_run=True


,setting,value
0,dry_run,True
1,update_labels,True
2,update_descriptions,True
3,update_aliases,True
4,update_statements,True
5,remove_obsolete_aliases,False
6,remove_obsolete_statements,False
7,delete_missing_items,False
8,metadata_sync_mode,merge
9,default_language,en


## Section 4: Property Mapping Dictionary

Wikibase properties are **created manually** in the target instance (Special:NewProperty).
Once created, fill in their PIDs below. `PROPERTY_MAP` lives in
`src/config.py` so it is reusable outside the notebook, but it is shown and
editable here since it is the single most important piece of per-deployment
configuration.

The pipeline **never invents a PID or guesses a datatype**: entries default
to `pid=None` and are reported as blocking issues if used in the OWL file
but left unmapped.

In [4]:
# Edit src/config.py's PROPERTY_MAP (imported as PROPERTY_MAP above) to add real PIDs,
# or override individual entries here before validation. Example:
#
# PROPERTY_MAP["source"]["pid"] = "P12"
# PROPERTY_MAP["ontology_iri"]["pid"] = "P24"

PROPERTY_MAPPINGS = build_property_mappings(PROPERTY_MAP)
pd.DataFrame([
    {"predicate": key, "pid": m.pid or "<NOT SET>", "wikibase_datatype": m.wikibase_datatype, "role": m.role}
    for key, m in PROPERTY_MAPPINGS.items()
])

,predicate,pid,wikibase_datatype,role
0,rdf:type,P16,wikibase-item,instance_of
1,rdfs:subClassOf,P18,wikibase-item,subclass_of
2,source,P2,string,None
3,wikidata_uri,P3,url,None
4,orkg_id,P1,external-id,None
5,ontology_iri,P17,url,identity
6,hasDataFormatSpecification,P4,wikibase-item,None
7,hasDataItem,P5,wikibase-item,None
8,hasDataModel,P6,wikibase-item,None
9,hasInterchangeFormat,<NOT SET>,wikibase-item,None


## Section 5: Load and Parse the OWL File

`load_ontology_graph` validates that the file exists and is non-empty,
auto-detects the RDF serialization from the file extension (falling back to
a few common formats), and returns a populated `rdflib.Graph`. The graph is
used directly by every later section -- it is never converted to CSV.

In [5]:
assert CONFIG.owl_file_path.exists(), f"OWL file not found: {CONFIG.owl_file_path}"

OWL_FILE_HASH = compute_file_hash(CONFIG.owl_file_path)
GRAPH = load_ontology_graph(CONFIG.owl_file_path)

assert len(GRAPH) > 0, "Parsed graph is empty; the OWL file may be malformed."
LOGGER.info("Parsed %s: %d triples (sha256=%s)", CONFIG.owl_file_path.name, len(GRAPH), OWL_FILE_HASH[:12])

print(f"OWL file:      {CONFIG.owl_file_path}")
print(f"File hash:     {OWL_FILE_HASH}")
print(f"Total triples: {len(GRAPH)}")
print(f"Namespaces:    {dict(GRAPH.namespaces())}")

wptmp:entity#data importing does not look like a valid URI, trying to serialize this will break.


wptmp:entity#export data does not look like a valid URI, trying to serialize this will break.


2026-07-13T21:20:18 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync.ontology_parser | Parsed /Users/jamaleldemashki/Desktop/owl-wikibase-sync/owl-wikibase-sync/data/slr_reviewed.owl using format=xml (14560 triples)


2026-07-13T21:20:18 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync | Parsed slr_reviewed.owl: 14560 triples (sha256=c2e8856cded4)


OWL file:      /Users/jamaleldemashki/Desktop/owl-wikibase-sync/owl-wikibase-sync/data/slr_reviewed.owl
File hash:     c2e8856cded4624ad2c6a51334c9636cdd14ca69c7b89c33a2761d57be872c4e
Total triples: 14560
Namespaces:    {'brick': rdflib.term.URIRef('https://brickschema.org/schema/Brick#'), 'csvw': rdflib.term.URIRef('http://www.w3.org/ns/csvw#'), 'dc': rdflib.term.URIRef('http://purl.org/dc/elements/1.1/'), 'dcat': rdflib.term.URIRef('http://www.w3.org/ns/dcat#'), 'dcmitype': rdflib.term.URIRef('http://purl.org/dc/dcmitype/'), 'dcterms': rdflib.term.URIRef('http://purl.org/dc/terms/'), 'dcam': rdflib.term.URIRef('http://purl.org/dc/dcam/'), 'doap': rdflib.term.URIRef('http://usefulinc.com/ns/doap#'), 'foaf': rdflib.term.URIRef('http://xmlns.com/foaf/0.1/'), 'geo': rdflib.term.URIRef('http://www.opengis.net/ont/geosparql#'), 'odrl': rdflib.term.URIRef('http://www.w3.org/ns/odrl/2/'), 'org': rdflib.term.URIRef('http://www.w3.org/ns/org#'), 'prof': rdflib.term.URIRef('http://www.w3.org/ns

In [6]:
# Sample of parsed triples, for a quick sanity check.
sample_rows = [
    {"subject": str(s), "predicate": str(p), "object": str(o)}
    for s, p, o in list(GRAPH)[:15]
]
pd.DataFrame(sample_rows)

,subject,predicate,object
0,http://tib.eu/slr#contribution89,http://tib.eu/slr#mentions,http://tib.eu/slr#http://purl.obolibrary.org/o...
1,http://tib.eu/slr#contribution123,http://tib.eu/slr#mentions,http://tib.eu/slr#http://purl.obolibrary.org/o...
2,http://tib.eu/slr#contribution79,http://tib.eu/slr#mentions,http://tib.eu/slr#http://purl.obolibrary.org/o...
3,wptmp:entity#export data,http://tib.eu/slr#source,curation
4,http://tib.eu/slr#contribution64,http://tib.eu/slr#orkg_id,R1357018
5,http://tib.eu/slr#http://purl.obolibrary.org/o...,http://tib.eu/slr#hasDataItem,http://webprotege.stanford.edu/R81IXS3kOGNwYWY...
6,http://tib.eu/slr#contribution64,http://tib.eu/slr#mentions,http://tib.eu/slr#http://purl.obolibrary.org/o...
7,http://tib.eu/slr#http://purl.obolibrary.org/o...,http://www.w3.org/2000/01/rdf-schema#label,database
8,http://tib.eu/slr#contribution29,http://tib.eu/slr#mentions,http://tib.eu/slr#http://purl.obolibrary.org/o...
9,http://tib.eu/slr#contribution128,http://tib.eu/slr#mentions,http://tib.eu/slr#http://purl.obolibrary.org/o...


## Section 6: Ontology Investigation and Classification

Classify every resource in the graph and extract, per entity: canonical id,
original id, local name, entity type, labels/descriptions/aliases by
language, and outgoing statements. This is implemented in
`src/ontology_parser.py` via `normalize_resource_identifier`,
`get_local_name`, `classify_resource`, `classify_predicate`,
`get_language_values`, `extract_entity_metadata`, and `extract_statements`.

Identifiers are **not** assumed to be consistently formatted -- the parser
handles `#Contribution`, `#http://.../BFO_0000015`, and
`http://.../BFO_0000015` as referring to the same canonical resource when
applicable (see `src/identifiers.py`).

In [7]:
PARSER_CONFIG = ParserConfig(
    default_language=CONFIG.default_language,
    alias_local_names=frozenset(name.lower() for name in CONFIG.alias_local_names),
)

ENTITIES = collect_syncable_entities(GRAPH, PARSER_CONFIG)
PROPERTY_LABEL_INDEX = build_property_label_index(GRAPH)
PREDICATE_USAGE = collect_predicate_usage(ENTITIES)

LOGGER.info("Classified %d syncable entities (classes + named individuals)", len(ENTITIES))

entity_type_counts = pd.Series([e.entity_type for e in ENTITIES]).value_counts()
entity_type_counts.to_frame("count")

2026-07-13T21:20:18 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync.ontology_parser | owl:Thing is used as a superclass in this ontology; synthesized a syncable entity for it (http://www.w3.org/2002/07/owl#Thing).


2026-07-13T21:20:18 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync | Classified 964 syncable entities (classes + named individuals)


,count
NamedIndividual,956
OwlClass,8


In [8]:
# Human-readable view of unmapped predicates, resolved against declared property labels
# where available (useful for WebProtege-style opaque object property ids).
unmapped_rows = []
for predicate_key, usage in sorted(PREDICATE_USAGE.items()):
    if predicate_key in PROPERTY_MAPPINGS:
        continue
    label = PROPERTY_LABEL_INDEX.get(predicate_key, "")
    unmapped_rows.append({
        "predicate_key": predicate_key,
        "resolved_label": label,
        "usage_count": usage["count"],
        "value_kinds": ", ".join(sorted(usage["value_kinds"])),
    })
pd.DataFrame(unmapped_rows).sort_values("usage_count", ascending=False) if unmapped_rows else "No unmapped predicates."

,predicate_key,resolved_label,usage_count,value_kinds
0,R7NnYCnkiXns2ntVncqu1F0,,137,resource
1,sameAs,,17,resource
2,seeAlso,,4,literal


## Section 7: Ontology Statistics

Aggregate statistics over the parsed ontology, saved to
`reports/ontology_statistics.json` (and a flattened `.csv`).

In [9]:
STATISTICS = compute_ontology_statistics(
    GRAPH, ENTITIES, PREDICATE_USAGE, PROPERTY_MAPPINGS, frozenset(n.lower() for n in CONFIG.alias_local_names)
)
save_ontology_statistics(CONFIG.report_dir, STATISTICS)

display_rows = [
    {"metric": k, "value": (json.dumps(v) if isinstance(v, (dict, list)) else v)}
    for k, v in STATISTICS.items()
]
pd.DataFrame(display_rows)

,metric,value
0,total_triples,14560
1,unique_subjects,995
2,unique_predicates,25
3,unique_objects,3529
4,owl_classes,7
5,named_individuals,956
6,object_properties,13
7,datatype_properties,0
8,annotation_properties,7
9,blank_nodes,1


## Section 8: Data Quality and Preflight Validation

Preflight checks run entirely offline (no Wikibase connection required) and
classify findings as `INFO` / `WARNING` / `ERROR` / `BLOCKING`. Problematic
data is **reported, never silently discarded**. If any `BLOCKING` finding
exists and `STOP_ON_VALIDATION_ERROR=True`, live synchronization will refuse
to run later in this notebook (dry-run planning is unaffected, so the plan
itself always remains inspectable).

In [10]:
QUALITY_FINDINGS = validate_ontology_quality(
    ENTITIES, PROPERTY_MAPPINGS, CONFIG.default_language, CONFIG.supported_languages
)
MAPPING_FINDINGS, MAPPING_SUMMARY = validate_property_map(PROPERTY_MAPPINGS, PREDICATE_USAGE)
ALL_PREFLIGHT_FINDINGS = QUALITY_FINDINGS + MAPPING_FINDINGS

severity_counts = summarize_findings(ALL_PREFLIGHT_FINDINGS)
print("Preflight finding counts by severity:", severity_counts)
print("\nProperty-mapping summary:")
for key in ("mapped_predicates", "unmapped_predicates", "unused_mappings", "predicates_missing_pid"):
    print(f"  {key}: {len(MAPPING_SUMMARY[key])}")

HAS_BLOCKING_FINDINGS = has_blocking_findings(ALL_PREFLIGHT_FINDINGS)
print(f"\nHas BLOCKING findings: {HAS_BLOCKING_FINDINGS}")

save_validation_report(CONFIG.report_dir, ALL_PREFLIGHT_FINDINGS, {
    "severity_counts": severity_counts,
    "property_mapping_summary": MAPPING_SUMMARY,
    "stage": "preflight",
})

pd.DataFrame([f.as_dict() for f in ALL_PREFLIGHT_FINDINGS]).sort_values("severity") if ALL_PREFLIGHT_FINDINGS else "No findings."

Preflight finding counts by severity: {'INFO': 623, 'WARNING': 474, 'ERROR': 0, 'BLOCKING': 0}

Property-mapping summary:
  mapped_predicates: 17
  unmapped_predicates: 3
  unused_mappings: 2
  predicates_missing_pid: 0

Has BLOCKING findings: False


,check,severity,message,source_identifier,details
548,malformed_uri,INFO,Original identifier 'http://tib.eu/slr#http://...,http://purl.obolibrary.org/obo/iao_000002754,{}
488,malformed_uri,INFO,Original identifier 'http://tib.eu/slr#http://...,http://purl.obolibrary.org/obo/iao_000002728,{}
874,malformed_uri,INFO,Original identifier 'http://tib.eu/slr#http://...,http://www.ebi.ac.uk/swo/swo_000000168,{}
491,malformed_uri,INFO,Original identifier 'http://tib.eu/slr#http://...,http://purl.obolibrary.org/obo/iao_000002729,{}
493,malformed_uri,INFO,Original identifier 'http://tib.eu/slr#http://...,http://purl.obolibrary.org/obo/iao_00000273,{}
...,...,...,...,...,...
487,invalid_url_value,WARNING,Value 'None' for predicate 'wikidata_uri' is n...,http://purl.obolibrary.org/obo/iao_000002727,"{'predicate': 'wikidata_uri', 'value': 'None'}"
489,predicate_without_mapping,WARNING,Entity 'http://purl.obolibrary.org/obo/iao_000...,http://purl.obolibrary.org/obo/iao_000002728,{'predicate': 'R7NnYCnkiXns2ntVncqu1F0'}
490,invalid_url_value,WARNING,Value 'None' for predicate 'wikidata_uri' is n...,http://purl.obolibrary.org/obo/iao_000002728,"{'predicate': 'wikidata_uri', 'value': 'None'}"
452,invalid_url_value,WARNING,Value 'None' for predicate 'wikidata_uri' is n...,http://purl.obolibrary.org/obo/bfo_000001599,"{'predicate': 'wikidata_uri', 'value': 'None'}"


## Section 9: Wikibase Credentials and Connection

Credentials are loaded from `.env` (see `.env.example`) via
`load_wikibase_credentials`. The connection test checks API availability,
authentication, bot permissions, and (optionally) SPARQL endpoint
availability. Dry-run planning only needs `api_available` -- authentication
is only required for live writes.

In [11]:
CREDENTIALS = load_wikibase_credentials()
print("Credential summary (redacted):", CREDENTIALS.describe_safe())

CLIENT = WikibaseClient(CREDENTIALS, CONFIG)
CONNECTION_STATUS = CLIENT.connect(require_write=not CONFIG.dry_run)

print(f"\napi_available:     {CONNECTION_STATUS.api_available}")
print(f"authenticated:     {CONNECTION_STATUS.authenticated}")
print(f"bot_permissions:   {CONNECTION_STATUS.bot_permissions}")
print(f"sparql_available:  {CONNECTION_STATUS.sparql_available}")
if CONNECTION_STATUS.errors:
    print("errors:")
    for error in CONNECTION_STATUS.errors:
        print(f"  - {error}")

if not CONFIG.dry_run and not CONNECTION_STATUS.ready_for_live_writes:
    raise RuntimeError(
        "DRY_RUN is False but the Wikibase connection is not ready for live writes. "
        "Fix .env credentials or set CONFIG.dry_run back to True."
    )

# Only use the client for read-based planning if the API is actually reachable;
# otherwise the planner falls back to cache-only resolution (see Section 12).
READ_CLIENT = CLIENT if CONNECTION_STATUS.ready_for_dry_run else None

Credential summary (redacted): {'wikibase_url': 'https://owl-wikibase-sync.wikibase.cloud', 'api_url': 'https://owl-wikibase-sync.wikibase.cloud/w/api.php', 'sparql_url': 'https://owl-wikibase-sync.wikibase.cloud/query/sparql', 'bot_username': 'Jamaleldemashki@owlwikisyncer', 'bot_password': '<redacted>'}



api_available:     True
authenticated:     True
bot_permissions:   True
sparql_available:  True


## Section 10: Persistent Entity Lookup Cache

`cache/entity_lookup.json` is the backbone of idempotency: canonical
ontology id &rarr; Wikibase QID. It is loaded here (with automatic recovery
from a malformed file, quarantined rather than silently deleted) and backed
up before this run makes any changes to it.

In [12]:
CACHE = EntityLookupCache.load(CONFIG.cache_dir / "entity_lookup.json", wikibase_url=CREDENTIALS.wikibase_url)
print(f"Cache belongs to current Wikibase instance: {CACHE.belongs_to_current_instance()}")
print(f"Cached entities: {len(CACHE.entities)}")

duplicate_qids = CACHE.find_duplicate_qids()
if duplicate_qids:
    print(f"WARNING: {len(duplicate_qids)} QIDs are mapped from more than one canonical id:")
    print(duplicate_qids)
else:
    print("No duplicate QID mappings found in the cache.")

if CACHE.entities:
    backup_path = CACHE.backup()
    print(f"Backed up existing cache to: {backup_path}")

pd.DataFrame(CACHE.as_dataframe_records()).head(20) if CACHE.entities else "Cache is currently empty (expected on a first run)."

Cache belongs to current Wikibase instance: True
Cached entities: 0
No duplicate QID mappings found in the cache.


'Cache is currently empty (expected on a first run).'

## Section 11: Wikibase Fallback Lookup

When an entity is not in the local cache, the pipeline queries Wikibase by
the configured identity property (`ontology_iri`, via SPARQL) **instead of**
searching by label. Zero matches &rarr; plan a new item. Exactly one match
&rarr; reuse it (and backfill the cache). More than one match &rarr; a
`CONFLICT`, which blocks that entity until resolved manually. This logic is
implemented once in `src/sync_planner.py::plan_entity_resolution` and used
by both dry-run and live execution -- this section only demonstrates it on a
single example so its behavior is visible before running the full plan.

In [13]:
IDENTITY_MAPPING = PROPERTY_MAPPINGS.get(CONFIG.identity_property_key)
if IDENTITY_MAPPING is None or not IDENTITY_MAPPING.pid:
    print(
        f"PROPERTY_MAP['{CONFIG.identity_property_key}']['pid'] is not set yet -- remote lookup by identity "
        "is disabled until it is configured. The local cache is still authoritative for entities it already knows."
    )
elif READ_CLIENT is not None and CREDENTIALS.sparql_url:
    example_entity = ENTITIES[0]
    matches = READ_CLIENT.find_qids_by_identity_value(IDENTITY_MAPPING.pid, example_entity.canonical_id)
    print(f"Example lookup for '{example_entity.canonical_id}': {matches or 'no matches (would be created)'}")
else:
    print("No live/SPARQL connection available; fallback lookup will be skipped during planning (cache-only mode).")

Example lookup for 'http://www.w3.org/2002/07/owl#Thing': no matches (would be created)


## Section 12: Two-Pass Synchronization Planning

**Pass A** resolves or plans creation of a Wikibase item for every OWL class
and named individual, and plans label/description/alias changes.
**Pass B** (run only once every entity has a resolved-or-placeholder QID)
plans statement additions, resolving resource-valued objects to their
(possibly not-yet-created) target items.

This produces the single `PlanResult` that both dry-run display (Section 13)
and live execution (Section 15) consume -- there is no separate "live
planning" code path.

In [14]:
PLAN = build_synchronization_plan(ENTITIES, PROPERTY_MAPPINGS, CACHE, READ_CLIENT, CONFIG)

action_counts = PLAN.action_counts()
LOGGER.info("Synchronization plan built: %d total actions", len(PLAN.actions))
pd.Series(action_counts).sort_values(ascending=False).to_frame("count")

2026-07-13T21:20:22 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync.sync_planner | Resolved 964 cache-miss entities against Wikibase using 20 batched SPARQL queries (batch_size=50).


2026-07-13T21:20:22 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync | Synchronization plan built: 15468 total actions


,count
ADD_STATEMENT,12300
CREATE_ITEM,964
UPDATE_LABEL,963
ADD_ALIAS,770
ERROR,311
SKIP_UNMAPPED_PROPERTY,158
ADD_DESCRIPTION,2


## Section 13: Dry-Run Plan Review and Export

The plan is exported to `reports/dry_run_plan.json` and
`reports/dry_run_plan.csv` regardless of `DRY_RUN`, since it is useful audit
output either way. New items are shown with the placeholder QID `<NEW_ITEM>`
-- the pipeline never invents a real QID during planning.

In [15]:
DRY_RUN_PAYLOAD = save_dry_run_plan(CONFIG.report_dir, PLAN.actions)

plan_df = pd.DataFrame([a.as_dict() for a in PLAN.actions])
print(f"Total planned actions: {len(plan_df)}")
print("\nBy action type:")
print(plan_df["action"].value_counts())
print("\nBy severity:")
print(plan_df["severity"].value_counts())

plan_df.head(30)

Total planned actions: 15468

By action type:
action
ADD_STATEMENT             12300
CREATE_ITEM                 964
UPDATE_LABEL                963
ADD_ALIAS                   770
ERROR                       311
SKIP_UNMAPPED_PROPERTY      158
ADD_DESCRIPTION               2
Name: count, dtype: int64

By severity:
severity
INFO       14999
ERROR        311
WARNING      158
Name: count, dtype: int64


,action,entity_type,source_identifier,qid,label,pid,property_name,old_value,new_value,reason,severity,target_source_identifier
0,CREATE_ITEM,OwlClass,http://www.w3.org/2002/07/owl#Thing,<NEW_ITEM>,Thing,,,,,no existing Wikibase item found,INFO,
1,UPDATE_LABEL,OwlClass,http://www.w3.org/2002/07/owl#Thing,<NEW_ITEM>,Thing,,label@en,,Thing,label missing for this language,INFO,
2,CREATE_ITEM,OwlClass,http://tib.eu/slr#Contribution,<NEW_ITEM>,Contribution,,,,,no existing Wikibase item found,INFO,
3,CREATE_ITEM,NamedIndividual,http://tib.eu/slr#contribution1,<NEW_ITEM>,Struck et al. - 1984 - EVALUATION OF OPERATION...,,,,,no existing Wikibase item found,INFO,
4,UPDATE_LABEL,NamedIndividual,http://tib.eu/slr#contribution1,<NEW_ITEM>,Struck et al. - 1984 - EVALUATION OF OPERATION...,,label@en,,Struck et al. - 1984 - EVALUATION OF OPERATION...,label missing for this language,INFO,
5,CREATE_ITEM,NamedIndividual,http://tib.eu/slr#contribution10,<NEW_ITEM>,Chang - 2003 - A knowledge base approach to as...,,,,,no existing Wikibase item found,INFO,
6,UPDATE_LABEL,NamedIndividual,http://tib.eu/slr#contribution10,<NEW_ITEM>,Chang - 2003 - A knowledge base approach to as...,,label@en,,Chang - 2003 - A knowledge base approach to as...,label missing for this language,INFO,
7,CREATE_ITEM,NamedIndividual,http://tib.eu/slr#contribution100,<NEW_ITEM>,Baalbergen et al. - 2018 - Integrated collabor...,,,,,no existing Wikibase item found,INFO,
8,UPDATE_LABEL,NamedIndividual,http://tib.eu/slr#contribution100,<NEW_ITEM>,Baalbergen et al. - 2018 - Integrated collabor...,,label@en,,Baalbergen et al. - 2018 - Integrated collabor...,label missing for this language,INFO,
9,CREATE_ITEM,NamedIndividual,http://tib.eu/slr#contribution101,<NEW_ITEM>,Van Gent et al. - 2018 - A Critical Look at De...,,,,,no existing Wikibase item found,INFO,


In [16]:
# Rows that need attention before a live run: conflicts, unresolved objects, errors, unmapped properties.
attention_types = {"CONFLICT", "UNRESOLVED_OBJECT", "ERROR", "SKIP_UNMAPPED_PROPERTY"}
attention_df = plan_df[plan_df["action"].isin(attention_types)]
print(f"{len(attention_df)} rows require attention before switching to live mode.")
attention_df.head(30)

469 rows require attention before switching to live mode.


,action,entity_type,source_identifier,qid,label,pid,property_name,old_value,new_value,reason,severity,target_source_identifier
10712,SKIP_UNMAPPED_PROPERTY,NamedIndividual,http://purl.obolibrary.org/obo/bfo_00000151,<NEW_ITEM>,Abaqus wrapping,,R7NnYCnkiXns2ntVncqu1F0,,http://webprotege.stanford.edu/R9Zjpxx0yUl7AHb...,predicate 'R7NnYCnkiXns2ntVncqu1F0' has no con...,WARNING,
10714,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_00000151,<NEW_ITEM>,Abaqus wrapping,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10721,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_000001510,<NEW_ITEM>,airworthiness testing,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10732,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015101,<NEW_ITEM>,flight-test preparation,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10738,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015102,<NEW_ITEM>,Foresight Scenarios (process),P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10743,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015103,<NEW_ITEM>,format transforming of semantic information,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10748,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015104,<NEW_ITEM>,FEM model generation,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10754,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015105,<NEW_ITEM>,graph-based support in the design problem form...,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10759,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015106,<NEW_ITEM>,ground test,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,
10766,ERROR,NamedIndividual,http://purl.obolibrary.org/obo/bfo_0000015107,<NEW_ITEM>,handling concepts and relationships between co...,P3,wikidata_uri,,None,conversion failed: not_a_uri:'None',ERROR,


## Section 14: Safety Gate

An explicit, impossible-to-miss gate: live writes only happen if `DRY_RUN`
is `False` **and** preflight validation did not surface a `BLOCKING` finding
under `STOP_ON_VALIDATION_ERROR`. Nothing upstream of this cell has written
anything to Wikibase.

In [17]:
if CONFIG.dry_run:
    print("DRY_RUN is True: dry-run completed. No Wikibase changes were committed.")
    print(f"Review reports/dry_run_plan.csv ({len(PLAN.actions)} rows) before setting CONFIG.dry_run = False.")
    PROCEED_WITH_LIVE_SYNC = False
elif HAS_BLOCKING_FINDINGS and CONFIG.stop_on_validation_error:
    print("Refusing to run live synchronization: preflight validation reported BLOCKING findings")
    print("and STOP_ON_VALIDATION_ERROR is True. Resolve them (see Section 8) or adjust the policy.")
    PROCEED_WITH_LIVE_SYNC = False
else:
    print("DRY_RUN is False and validation allows live execution. Proceeding to Section 15.")
    PROCEED_WITH_LIVE_SYNC = True

DRY_RUN is True: dry-run completed. No Wikibase changes were committed.
Review reports/dry_run_plan.csv (15468 rows) before setting CONFIG.dry_run = False.


## Section 15: Live Synchronization Execution

Executes exactly the plan built in Section 12: every `CREATE_ITEM` action
first (so all entities have real QIDs), then metadata updates, then
`ADD_STATEMENT` actions. Every successful write is immediately followed by a
cache/checkpoint save (`src/synchronizer.py`), so an interruption here can be
safely resumed by re-running this notebook -- reruns re-evaluate live/cache
state rather than blindly replaying writes.

In [18]:
STATE = SynchronizationState.load_or_start(
    CONFIG.cache_dir / "synchronization_state.json",
    run_id=RUN_ID,
    owl_file_hash=OWL_FILE_HASH,
    config_hash=CONFIG.config_hash(),
    wikibase_url=CREDENTIALS.wikibase_url,
    start_time=RUN_TIMESTAMP,
)

if PROCEED_WITH_LIVE_SYNC:
    SYNC_RESULT = execute_synchronization(
        PLAN, ENTITIES, PROPERTY_MAPPINGS, CACHE, CLIENT, STATE, CONFIG, RUN_ID, RUN_TIMESTAMP
    )
    LOGGER.info("Synchronization finished with status=%s", SYNC_RESULT.status)
    pd.Series(SYNC_RESULT.as_dict()).to_frame("value")
else:
    from src.models import SynchronizationResult
    SYNC_RESULT = SynchronizationResult(run_id=RUN_ID, status="skipped_dry_run")
    print("Skipped live execution (see Section 14).")

Skipped live execution (see Section 14).


## Section 16: Post-Synchronization Validation

After a live run, verify pipeline invariants: every processed entity has a
QID, no two canonical ids share a QID, and every unresolved
entity/statement/conflict from the executed plan is surfaced rather than
silently dropped. In dry-run mode this section reports the plan's
not-yet-executed unresolved items instead, since there is no live outcome to
verify yet.

In [19]:
POST_SYNC_FINDINGS = validate_post_synchronization(ENTITIES, CACHE, PLAN.actions)
post_sync_summary = summarize_findings(POST_SYNC_FINDINGS)
print("Post-synchronization finding counts by severity:", post_sync_summary)

save_validation_report(CONFIG.report_dir, POST_SYNC_FINDINGS, {
    "severity_counts": post_sync_summary,
    "stage": "post_synchronization" if PROCEED_WITH_LIVE_SYNC else "dry_run_plan_only",
})

pd.DataFrame([f.as_dict() for f in POST_SYNC_FINDINGS]).sort_values("severity") if POST_SYNC_FINDINGS else "No findings."

Post-synchronization finding counts by severity: {'INFO': 0, 'WARNING': 158, 'ERROR': 1275, 'BLOCKING': 0}


,check,severity,message,source_identifier,details
0,missing_qid_after_sync,ERROR,Entity 'http://www.w3.org/2002/07/owl#Thing' h...,http://www.w3.org/2002/07/owl#Thing,{}
854,missing_qid_after_sync,ERROR,Entity 'http://webprotege.stanford.edu/R9r5F1Y...,http://webprotege.stanford.edu/R9r5F1YMuUNiIAq...,{}
853,missing_qid_after_sync,ERROR,Entity 'http://webprotege.stanford.edu/R9iknGT...,http://webprotege.stanford.edu/R9iknGTWhtMbUDN...,{}
852,missing_qid_after_sync,ERROR,Entity 'http://webprotege.stanford.edu/R9haSQ2...,http://webprotege.stanford.edu/R9haSQ2mm68hfLs...,{}
851,missing_qid_after_sync,ERROR,Entity 'http://webprotege.stanford.edu/R9dYtQ6...,http://webprotege.stanford.edu/R9dYtQ6BrGXpMHt...,{}
...,...,...,...,...,...
1180,unresolved_action_skip_unmapped_property,WARNING,SKIP_UNMAPPED_PROPERTY: predicate 'R7NnYCnkiXn...,http://purl.obolibrary.org/obo/iao_000002726,{}
1008,unresolved_action_skip_unmapped_property,WARNING,SKIP_UNMAPPED_PROPERTY: predicate 'R7NnYCnkiXn...,http://purl.obolibrary.org/obo/bfo_0000015145,{}
1179,unresolved_action_skip_unmapped_property,WARNING,SKIP_UNMAPPED_PROPERTY: predicate 'seeAlso' ha...,http://purl.obolibrary.org/obo/iao_000002724,{}
1310,unresolved_action_skip_unmapped_property,WARNING,SKIP_UNMAPPED_PROPERTY: predicate 'R7NnYCnkiXn...,http://www.ebi.ac.uk/swo/swo_000000142,{}


## Section 17: Reports, Checkpoints, and Logging Summary

Writes the remaining machine-readable reports
(`synchronization_report.json`, `unresolved_resources.csv`) and confirms
what is now on disk under `reports/`, `cache/`, and `logs/`.

In [20]:
save_synchronization_report(CONFIG.report_dir, SYNC_RESULT, PLAN.actions)
unresolved_rows = save_unresolved_resources(CONFIG.report_dir, PLAN.actions)
print(f"{len(unresolved_rows)} unresolved/conflicting/error rows written to reports/unresolved_resources.csv")

print("\nReport directory contents:")
for report_file in sorted(CONFIG.report_dir.glob("*")):
    print(f"  {report_file.name}  ({report_file.stat().st_size} bytes)")

print("\nCache directory contents:")
for cache_file in sorted(CONFIG.cache_dir.glob("*")):
    print(f"  {cache_file.name}  ({cache_file.stat().st_size} bytes)")

469 unresolved/conflicting/error rows written to reports/unresolved_resources.csv

Report directory contents:
  .gitkeep  (0 bytes)
  dry_run_plan.csv  (3136338 bytes)
  dry_run_plan.json  (7481670 bytes)
  ontology_statistics.csv  (858 bytes)
  ontology_statistics.json  (1026 bytes)
  synchronization_report.json  (629 bytes)
  unresolved_resources.csv  (97715 bytes)
  validation_report.json  (409243 bytes)

Cache directory contents:
  .gitkeep  (0 bytes)


## Section 18: Final Summary

In [21]:
traffic_stats = CLIENT.get_traffic_stats()

summary = {
    "run_id": RUN_ID,
    "dry_run": CONFIG.dry_run,
    "status": SYNC_RESULT.status,
    "entities_parsed": len(ENTITIES),
    "items_created": SYNC_RESULT.items_created,
    "items_reused": SYNC_RESULT.items_reused,
    "labels_updated": SYNC_RESULT.labels_updated,
    "descriptions_updated": SYNC_RESULT.descriptions_updated,
    "aliases_added": SYNC_RESULT.aliases_added,
    "statements_added": SYNC_RESULT.statements_added,
    "statements_skipped_unchanged": SYNC_RESULT.statements_skipped_unchanged,
    "unresolved_entities": SYNC_RESULT.unresolved_entities,
    "unresolved_properties": SYNC_RESULT.unresolved_properties,
    "conflicts": SYNC_RESULT.conflicts,
    "errors": SYNC_RESULT.errors,
    "api_calls": traffic_stats["api_calls"],
    "retries": traffic_stats["retries"],
}

LOGGER.info("Run summary: %s", json.dumps(summary))
pd.Series(summary).to_frame("value")

2026-07-13T21:20:22 | INFO     | run=20260713T192017-8fa7c2dc | owl_wikibase_sync | Run summary: {"run_id": "20260713T192017-8fa7c2dc", "dry_run": true, "status": "skipped_dry_run", "entities_parsed": 964, "items_created": 0, "items_reused": 0, "labels_updated": 0, "descriptions_updated": 0, "aliases_added": 0, "statements_added": 0, "statements_skipped_unchanged": 0, "unresolved_entities": 0, "unresolved_properties": 0, "conflicts": 0, "errors": 0, "api_calls": 23, "retries": 0}


,value
run_id,20260713T192017-8fa7c2dc
dry_run,True
status,skipped_dry_run
entities_parsed,964
items_created,0
items_reused,0
labels_updated,0
descriptions_updated,0
aliases_added,0
statements_added,0
